# 01 · Sparse reconstruction of the radiator

Phone video → sharp frames → structure-from-motion (pycolmap, CPU) → interactive 3D view.

The logic lives in `src/trenchscan/`; this notebook only calls it and visualises results. Edit the package, and `autoreload` picks up the changes without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np, yaml
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # change to "vscode" if running inside VS Code

from trenchscan import frames, sfm, viz

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VIDEO = ROOT / "data/raw/VID20261023102419.mp4"   # clip 3: the clean arc
WORK = ROOT / "outputs/clip3"
GT = yaml.safe_load(open(ROOT / "data/ground_truth/radiator_2026-10-23.yaml"))
GT["measured"]

## 1 · Frames and blur filter
Extract at 4 fps, then keep the sharpest 75 % (variance of the Laplacian).

In [ ]:
all_frames = frames.extract_frames(VIDEO, WORK / "raw", fps=4)
kept, scores = frames.filter_sharp(all_frames, WORK / "frames", keep_fraction=0.75)
thr = np.percentile(scores, 25)

fig = go.Figure(go.Scatter(y=scores, mode="lines+markers", marker=dict(size=4,
        color=np.where(scores >= thr, "#1D9E75", "#E24B4A")), name="sharpness"))
fig.add_hline(y=thr, line_dash="dash", annotation_text="keep above")
fig.update_layout(height=280, xaxis_title="frame", yaxis_title="Laplacian variance", margin=dict(t=20))
fig.show()
print(f"{len(all_frames)} extracted, {len(kept)} kept")

In [ ]:
# Sharpest vs blurriest frames
order = np.argsort(scores)
fig, ax = plt.subplots(2, 4, figsize=(12, 7))
for a, i in zip(ax[0], order[-4:][::-1]): a.imshow(plt.imread(all_frames[i])); a.set_title(f"sharp {scores[i]:.0f}")
for a, i in zip(ax[1], order[:4]):        a.imshow(plt.imread(all_frames[i])); a.set_title(f"blurry {scores[i]:.0f}")
for a in ax.flat: a.axis("off")
plt.tight_layout()

## 2 · Structure-from-motion
SIFT → sequential matching → incremental mapping. About 1 min on a laptop CPU; cached in `outputs/` afterwards (`overwrite=True` to rerun).

In [ ]:
rec = sfm.run_sfm(WORK / "frames", WORK / "sparse")
sfm.summary(rec, n_input=len(kept))

## 3 · Look at it
Red pyramids = where the phone was for each registered frame. Drag to rotate.

In [ ]:
cloud = sfm.clean(sfm.to_cloud(rec))
viz.plot_cloud(cloud, title=f"Clip 3 · {len(cloud.xyz)} points, {len(cloud.cam_centers)} cameras").show()

## 4 · Where are the points? (texture check)
Points come from texture. The white tube surfaces should be sparse compared with edges, floor and skirting board — see `docs/FINDINGS.md`.
A quick way to see it: how many 3D points does each frame observe?

In [ ]:
obs = sorted(((im.name, im.num_points3D) for im in rec.images.values()))
fig = go.Figure(go.Bar(x=[n for n, _ in obs], y=[k for _, k in obs]))
fig.update_layout(height=280, xaxis_title="frame", yaxis_title="3D points seen", margin=dict(t=20))
fig.show()

## Your turn
Things worth trying (each is a one-line change above):
- `keep_fraction` 0.5 vs 0.9 — does a stricter blur filter register more or fewer frames?
- `fps=2` vs `fps=6` — fewer, more different frames vs more overlap.
- `overlap=5` vs `overlap=25` in `run_sfm` — matching speed vs robustness.

Log what you find in `docs/FINDINGS.md`.

**Next notebook (02):** metric scale from the bank card, then compare against `GT`.